# 环境配置

## 安装包管理工具

In [ ]:
%pip install uv

!uv --version

## 安装依赖

In [ ]:
!uv pip install datasets==4.0.0 transformers==4.53.1

In [ ]:
!uv add tokenizers==0.21.2

In [ ]:
!uv add torch==2.7.1

### 11.2.3 建立词表索引

In [ ]:
vocabulary = {}
for text in dataset:
  text = standardize(text)
  tokens = tokenize(text)
  for token in tokens:
    if token not in vocabulary:
      vocabulary[token] = len(vocabulary)

### 11.2.4 使用 TextVectorization 层

In [ ]:
import string

class Vectorizer:
    def standardize(self, text):
        text = text.lower()
        return "".join(char for char in text if char not in string.punctuation)

    def tokenize(self, text):
        text = self.standardize(text)
        return text.split()

    def make_vocabulary(self, dataset):
        self.vocabulary = {"": 0, "[UNK]": 1}
        for text in dataset:
            text = self.standardize(text)
            tokens = self.tokenize(text)
            for token in tokens:
                if token not in self.vocabulary:
                    self.vocabulary[token] = len(self.vocabulary)
        self.inverse_vocabulary = dict(
            (v, k) for k, v in self.vocabulary.items())

    def encode(self, text):
        text = self.standardize(text)
        tokens = self.tokenize(text)
        return [self.vocabulary.get(token, 1) for token in tokens]

    def decode(self, int_sequence):
        return " ".join(
            self.inverse_vocabulary.get(i, "[UNK]") for i in int_sequence)

vectorizer = Vectorizer()
dataset = [
    "I write, erase, rewrite",
    "Erase again, and then",
    "A poppy blooms.",
]
vectorizer.make_vocabulary(dataset)

In [ ]:
test_sentence = "I write, rewrite, and still rewrite again"
encoded_sentence = vectorizer.encode(test_sentence)
print(encoded_sentence)

In [ ]:
decoded_sentence = vectorizer.decode(encoded_sentence)
print(decoded_sentence)

In [22]:
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.trainers import WordLevelTrainer
from tokenizers.normalizers import Lowercase
from tokenizers.pre_tokenizers import Whitespace, Sequence,Punctuation

class TextVec:
  def __init__(self, max_tokens=None):
     # 加载预训练的分词器
    tokenizer = Tokenizer(WordLevel(unk_token="[UNK]"))
    tokenizer.enable_padding()
    tokenizer.normalizer = Lowercase()
    tokenizer.pre_tokenizer = Sequence(
        [
            Whitespace(),
            Punctuation(behavior='removed')
        ]
    )

    if max_tokens:
      tokenizer.enable_truncation(max_tokens)

    self.tokenizer = tokenizer
  

  def __getattr__(self, name):
    return getattr(self.tokenizer, name)

  def adapt(self, data):
    trainer = WordLevelTrainer(special_tokens=["[UNK]"])

    self.tokenizer.train_from_iterator(data, trainer=trainer, length=len(data))

In [12]:
tokenizer = TextVec()

dataset = [
    "I write, erase, rewrite",
    "Erase again, and then",
    "A poppy blooms.",
]
tokenizer.adapt(dataset)

#### 展示词汇表

In [13]:
tokenizer.get_vocab()

{'erase': 1,
 'again': 3,
 'then': 9,
 'a': 2,
 'blooms': 5,
 '[UNK]': 0,
 'poppy': 7,
 'rewrite': 8,
 'i': 6,
 'and': 4,
 'write': 10}

In [16]:
vocabulary = tokenizer.get_vocab()
test_sentence = "I write, rewrite, and still rewrite again"
encoded_sentence = tokenizer.encode(test_sentence).ids
print(encoded_sentence)

[6, 10, 8, 4, 0, 8, 3]


In [17]:
inverse_vocab = {v: k for k, v in tokenizer.get_vocab().items()}
decoded_sentence = " ".join(inverse_vocab[int(i)] for i in encoded_sentence)
print(decoded_sentence)

i write rewrite and [UNK] rewrite again


## 11.3 表示单词组的两种方法：集合和序列

### 11.3.1 准备 IMDB 影评数据

In [ ]:
!curl -O https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz

In [ ]:
!tar -xf aclImdb_v1.tar.gz

In [ ]:
!rm -r aclImdb/train/unsup

In [ ]:
!cat aclImdb/train/pos/4077_10.txt

In [ ]:
import os, pathlib, shutil, random

base_dir = pathlib.Path("aclImdb")
val_dir = base_dir / "val"
train_dir = base_dir / "train"
for category in ("neg", "pos"):
    os.makedirs(val_dir / category)
    files = os.listdir(train_dir / category)
    random.Random(1337).shuffle(files)
    num_val_samples = int(0.2 * len(files))
    val_files = files[-num_val_samples:]
    for fname in val_files:
        shutil.move(train_dir / category / fname, val_dir / category / fname)

In [ ]:
!uv pip install datasets==4.0.0

In [1]:
import os

import datasets
from torch.utils.data import DataLoader

def text_dataset_from_dir(dir):
    out = []
    for f in filter(lambda x: x.is_dir(), os.scandir(dir)):
        label = 1 if f.name == "pos" else 0
        v = datasets.load_dataset("text", data_dir=f.path).map(lambda x: {"label": label})["train"]
        out.append(v)

    ds =  datasets.concatenate_datasets(out).with_format("torch") 

    return DataLoader(ds, batch_size=32, num_workers=8) 

/Users/xiangminli/Workspaces/github.com/sammyne/Deep-Learning-with-Python-2ed-cn/chapter11/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train_ds = text_dataset_from_dir("aclImdb/train")

val_ds = text_dataset_from_dir("aclImdb/val")

test_ds = text_dataset_from_dir("aclImdb/test")

Generating train split: 2500 examples [00:00, 13428.36 examples/s]
Map: 100%|██████████| 2500/2500 [00:00<00:00, 74076.04 examples/s]
Generating train split: 2500 examples [00:00, 13357.42 examples/s]
Map: 100%|██████████| 2500/2500 [00:00<00:00, 77186.88 examples/s]


In [ ]:
for v in train_ds:
    inputs, targets = v["text"], v["label"]

    print("inputs.len", len(inputs))
    print('inputs[0]:', inputs[0])

    print("targets.shape: ", targets.shape)
    print("targets[0]: ", targets[0])
    print("targets[0].dtype: ", targets[0].dtype)

    break

inputs.len 32
inputs[0]: Story of a man who has unnatural feelings for a pig. Starts out with a opening scene that is a terrific example of absurd comedy. A formal orchestra audience is turned into an insane, violent mob by the crazy chantings of it's singers. Unfortunately it stays absurd the WHOLE time with no general narrative eventually making it just too off putting. Even those from the era should be turned off. The cryptic dialogue would make Shakespeare seem easy to a third grader. On a technical level it's better than you might think with some good cinematography by future great Vilmos Zsigmond. Future stars Sally Kirkland and Frederic Forrest can be seen briefly.
targets.shape:  torch.Size([32])
targets[0]:  tensor(0)


In [18]:
import torch

def multi_hot(labels, nlabels):
    """Convert labels to multi-hot encoding."""
    if isinstance(labels, list) and all(isinstance(label, list) for label in labels):
        out = []
        for v in labels:
          encoded = torch.zeros((nlabels,), dtype=torch.float32)
          out[v] = 1.0
          out.append(out)
        
        return torch.stack(out)

    out = torch.zeros((nlabels,), dtype=torch.float32)
    out[labels] = 1.0
    return out

In [23]:
tokenizer = TextVec(max_tokens=20000)

text_only_train_ds = train_ds.map(lambda v: v["text"])

AttributeError: 'DataLoader' object has no attribute 'map'